# Cross-Asset Higher-Order Dependence via an MPS Born Machine
**MFE Industry Project · AQS**

> *"Typical risk models summarize market relationships with covariance and volatility. Those are useful, but
> they can miss changes in higher-order dependence, tail co-movement, and regime complexity. An MPS
> representation may provide additional diagnostics about the structure of the joint return distribution."*

We fit an MPS to the joint return distribution of a **diversified, multi-source cross-asset book** and test
whether it recovers dependence structure a Gaussian covariance model is **provably blind to** — across the
*whole* higher-order battery, not just tails:

| order | metric | what it means |
|---|---|---|
| 2 (baseline) | correlation | linear co-movement (what covariance captures) |
| all | total correlation / mutual information | total & pairwise dependence incl. nonlinear |
| 3 | **co-skewness** | does an asset crash *hard* when the market crashes hard? (priced — Harvey-Siddique) |
| 4 | **co-kurtosis** | shared fat-tailed extremes |
| tail | **lower-tail dependence λ_L, asymmetry λ_L−λ_U** | joint-crash probability; Gaussian copula ⇒ 0 |
| nonlinear | **MI − Gaussian-implied MI** | dependence beyond the linear correlation |
| regime | **stress-vs-calm correlation** | diversification failure in crises (Longin-Solnik) |

Data + economic motivation: see `crossasset_economics_brief.md`. Ordering = **Fiedler** (our tests show it
wins on diversified/block-structured books).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, sys
here = os.getcwd()
ROOT = here if os.path.isdir(os.path.join(here, "src")) else os.path.dirname(here)
sys.path.insert(0, os.path.join(ROOT, "src"))
def P(rel): return os.path.join(ROOT, rel)
import numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import mutual_info_score
from mps_regime import data as D, mps, binning, dmrg, features as F

PQ = "data/combined_crossasset_prices.parquet"
D_SEL, SW_SEL, CHI_GRID = 12, 10, [8, 16, 24, 32]   # held-out chi ranking (fast)
D_FIN, SW_FIN = 24, 12                                # final production fit (accuracy)
CLASS = {"USMKT":"US equity","AAPL":"US equity","NVDA":"US equity","EFA":"Intl equity","EEM":"Intl equity",
         "TLT":"Rates","LQD":"Credit","HYG":"Credit","VNQ":"Real asset","GC":"Real asset","CL":"Real asset","BTC":"Crypto"}
print("config d_sel=%d d_fin=%d" % (D_SEL, D_FIN))

config d_sel=12 d_fin=24


## 1 · The portfolio — 12 assets, 6 classes, 4 data sources
US equity (CRSP: **USMKT** value-weighted index, **AAPL**, **NVDA**) · Intl equity (**EFA** dev, **EEM** EM) ·
Rates (**TLT** 20y+ UST) · Credit (**LQD** IG, **HYG** high-yield) · Real assets (**GC** gold futures, **CL**
oil futures, **VNQ** REITs) · Crypto (**BTC**, CoinMetrics). Sources: CRSP survivorship-free returns, DataBento
futures, CoinMetrics, ETF closes — aligned on common business days, 2014–2024.

In [2]:
rets = D.to_returns(pd.read_parquet(P(PQ)).astype(float), kind="log")
N = rets.shape[1]; T = len(rets)
print("panel:", rets.shape, "|", rets.index.min().date(), "->", rets.index.max().date())
corr = rets.corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(N)); ax.set_xticklabels(corr.columns, rotation=90); ax.set_yticks(range(N)); ax.set_yticklabels(corr.columns)
for i in range(N):
    for j in range(N): ax.text(j, i, "%.2f"%corr.values[i,j], ha="center", va="center", fontsize=6)
plt.colorbar(im, fraction=0.046); ax.set_title("Linear correlation (what covariance sees)"); plt.tight_layout()
plt.savefig(P("data/ca_corr.png"), dpi=110); plt.show()
iu = np.triu_indices(N, 1)
print("avg pairwise corr = %.3f" % corr.values[iu].mean())
print("TLT avg corr = %.3f (classic diversifier) | HYG avg corr = %.3f (trades like equity)" %
      (corr['TLT'].drop('TLT').mean(), corr['HYG'].drop('HYG').mean()))

panel: (2583, 12) | 2014-09-19 -> 2024-12-31


avg pairwise corr = 0.296
TLT avg corr = 0.006 (classic diversifier) | HYG avg corr = 0.446 (trades like equity)


**Already visible in the linear correlations:** TLT ≈ 0 (the diversifier), HYG ≈ equity (high-yield's
"bond" label is a normal-regime illusion), gold/oil/BTC low. But correlation is a *single average number* — the
questions below are what it hides.

## 2 · Fit the MPS — Fiedler ordering, χ chosen by held-out likelihood (anti-overfitting)
We rank χ by **out-of-sample** dependence captured (independent-baseline test-NLL − model test-NLL) so we pick
the χ that generalizes, not the one that memorizes. Then we fit the final model at that χ and d=24.

In [3]:
# REVIEW FIX 2026-07-14 (BUG-LOG #15): ordering + edges from TRAIN ONLY; test encoded via fit.encode
T0 = len(rets); perm0 = np.random.default_rng(0).permutation(T0); cut0 = int(0.7*T0)
tr, te = sorted(perm0[:cut0]), sorted(perm0[cut0:])
_b, _, order = binning.discretize(rets.iloc[tr], n_bins=6); rets = rets[order]   # Fiedler (train-only)
print("Fiedler chain order (train-only):", list(order))
codes_tr, edges_tr, _, _ = mps._discretize(rets.iloc[tr], D_SEL)
_arr_te = rets.iloc[te].to_numpy()
codes_te = np.column_stack([np.digitize(_arr_te[:, k], edges_tr[k]) for k in range(N)])
ind = 0.0
for k in range(N):
    p = np.bincount(codes_tr[:,k], minlength=D_SEL)+1.0; p/=p.sum(); ind += -np.log(p[codes_te[:,k]]).mean()
sel = []
best_chi, best = CHI_GRID[0], -1e9
for chi in CHI_GRID:
    f = dmrg.fit_dmrg(rets.iloc[tr], d=D_SEL, chi=chi, sweeps=SW_SEL)
    tc = [torch.tensor(x) for x in f.cores]
    nte = float(mps._nll(tc, torch.tensor(f.encode(rets.iloc[te]), dtype=torch.long)))
    ntr = float(mps._nll(tc, torch.tensor(f.encode(rets.iloc[tr]), dtype=torch.long)))
    dep = ind - nte; sel.append((chi, ntr, nte, nte-ntr, dep))
    if dep > best: best, best_chi = dep, chi
    print("  chi=%2d train=%.3f test=%.3f gap=%+.3f heldout_dep=%+.3f %s" % (chi,ntr,nte,nte-ntr,dep,"<--best" if chi==best_chi else ""))
print("SELECTED chi* = %d" % best_chi)
fit = dmrg.fit_dmrg(rets, d=D_FIN, chi=best_chi, sweeps=SW_FIN)
print("final fit: d=%d chi*=%d | corr-capture=%.0f%%" % (D_FIN, best_chi,
      fit.mps_correlation()[iu].mean()/rets.corr().to_numpy()[iu].mean()*100))

Fiedler chain order (train-only): ['GC', 'TLT', 'LQD', 'CL', 'VNQ', 'HYG', 'EFA', 'EEM', 'NVDA', 'AAPL', 'USMKT', 'BTC']


  chi= 8 train=25.032 test=29.330 gap=+4.298 heldout_dep=+0.488 <--best


  chi=16 train=22.342 test=29.865 gap=+7.524 heldout_dep=-0.047 


  chi=24 train=20.717 test=30.283 gap=+9.566 heldout_dep=-0.465 


  chi=32 train=19.552 test=30.545 gap=+10.994 heldout_dep=-0.727 
SELECTED chi* = 8


final fit: d=24 chi*=8 | corr-capture=73%


In [4]:
# overfitting curve
seldf = pd.DataFrame(sel, columns=["chi","train_nll","test_nll","gap","heldout_dep"]).set_index("chi")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(seldf.index, seldf.train_nll, "o-", label="train NLL"); ax[0].plot(seldf.index, seldf.test_nll, "s-", label="test NLL")
ax[0].set_xlabel("bond dimension χ"); ax[0].set_ylabel("NLL (lower=better)"); ax[0].legend(); ax[0].set_title("Train vs held-out NLL")
ax[1].plot(seldf.index, seldf.heldout_dep, "D-", color="green"); ax[1].axvline(best_chi, ls="--", c="k", lw=1)
ax[1].set_xlabel("bond dimension χ"); ax[1].set_ylabel("held-out dependence captured"); ax[1].set_title("Generalization (pick the peak = chi*)")
plt.tight_layout(); plt.savefig(P("data/ca_chi.png"), dpi=110); plt.show()
print(seldf.round(3).to_string())

     train_nll  test_nll     gap  heldout_dep
chi                                          
8       25.032    29.330   4.298        0.488
16      22.342    29.865   7.524       -0.047
24      20.717    30.283   9.566       -0.465
32      19.552    30.545  10.994       -0.727


## 2b · Our designed feature suite, read off this cross-asset MPS
The regime/tail-complexity features we built for the module (F1–F13), evaluated on this fitted model. Each is
a single scalar diagnostic of the joint distribution; the "economic reading" column flags where it carries a
market interpretation vs. where it is a purely statistical/structural quantity.

In [5]:
feat = [
  ("required bond dim @ 90% quality", F.bond_dimension_at_quality(fit), "structural: effective # of factors the joint needs (absorption-ratio analogue)"),
  ("entanglement entropy (mid cut)",  F.entanglement_entropy(fit),      "structural: coupling across the chain; regime complexity proxy"),
  ("mean mutual information",         F.mean_mutual_information(fit, adjacent=False), "economic: average pairwise dependence incl. nonlinear"),
  ("total correlation (multi-info)",  F.total_correlation(fit),         "economic: TOTAL dependence beyond pairwise (system-wide coupling)"),
  ("tail-state probability",          F.tail_state_probability(fit),    "economic: prob. of an all-assets-down corner (systemic co-crash)"),
  ("tail-dependence asymmetry",       F.tail_dependence_asymmetry(fit), "economic: crashes vs rallies cluster (downside > upside)"),
  ("correlation length (transfer)",   F.correlation_length(fit),        "structural: how far dependence propagates along the chain"),
  ("reconstruction error",            F.reconstruction_error(fit, rets),"statistical: fit quality (lower=better)"),
  ("compression ratio",               F.compression_ratio(fit),         "statistical: parameter efficiency of the tensor rep"),
]
fdf = pd.DataFrame([(n, round(float(v),4)) for n,v,_ in feat], columns=["feature","value"]).set_index("feature")
fdf["reading"] = [r for _,_,r in feat]
print(fdf.to_string())

                                   value                                                                         reading
feature                                                                                                                 
required bond dim @ 90% quality   7.7179  structural: effective # of factors the joint needs (absorption-ratio analogue)
entanglement entropy (mid cut)    2.0435                  structural: coupling across the chain; regime complexity proxy
mean mutual information           0.0936                           economic: average pairwise dependence incl. nonlinear
total correlation (multi-info)    5.4690               economic: TOTAL dependence beyond pairwise (system-wide coupling)
tail-state probability            0.0000                economic: prob. of an all-assets-down corner (systemic co-crash)
tail-dependence asymmetry         0.0542                        economic: crashes vs rallies cluster (downside > upside)
correlation length (transfer)   

## 3 · The higher-order battery — empirical truth vs Gaussian null vs MPS
Helper functions compute each dependence measure three ways: **empirical** (the ground truth from the data),
**Gaussian** (Monte-Carlo from the fitted covariance — the null a covariance model assumes), and **MPS**
(analytic from the fitted tensor network). If the MPS lands closer to empirical than the Gaussian does, it is
carrying genuinely higher-order information.

In [6]:
def emp_pair_metrics(X, q=0.10):
    Z = (X - X.mean(0)) / X.std(0); N = X.shape[1]
    CS = np.zeros((N,N)); CK = np.zeros((N,N)); LL = np.zeros((N,N)); UU = np.zeros((N,N)); MI = np.zeros((N,N))
    loq = np.quantile(X, q, 0); hiq = np.quantile(X, 1-q, 0)
    B = np.stack([np.digitize(X[:,k], np.quantile(X[:,k], np.linspace(0,1,9)[1:-1])) for k in range(N)], 1)
    for i in range(N):
        for j in range(N):
            if i==j: continue
            CS[i,j] = (Z[:,i]**2 * Z[:,j]).mean()               # co-skew (asymmetric: i's variance vs j's direction)
            CK[i,j] = (Z[:,i]**2 * Z[:,j]**2).mean()
            LL[i,j] = ((X[:,i]<loq[i]) & (X[:,j]<loq[j])).mean()/q
            UU[i,j] = ((X[:,i]>hiq[i]) & (X[:,j]>hiq[j])).mean()/q
            if i<j: MI[i,j]=MI[j,i]=mutual_info_score(B[:,i], B[:,j])
    return dict(coskew=CS, cokurt=CK, lamL=LL, lamU=UU, MI=MI)

def mps_pair_metrics(fit, q=0.10):
    N = fit.n_sites; bv = fit.bin_values; p1 = fit.one_site_marginals()
    mu = (p1*bv).sum(1); sd = np.sqrt((p1*bv**2).sum(1)-mu**2)
    CS = np.zeros((N,N)); CK = np.zeros((N,N)); MI = np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            if i==j: continue
            p2 = fit.two_site_marginal(i, j); zi=(bv[i]-mu[i])/sd[i]; zj=(bv[j]-mu[j])/sd[j]
            CS[i,j] = float((p2*np.outer(zi**2, zj)).sum())
            CK[i,j] = float((p2*np.outer(zi**2, zj**2)).sum())
            if i<j: MI[i,j]=MI[j,i]=float(F.mutual_information(fit,i,j))
    LL = fit.mps_tail_dependence(q)
    return dict(coskew=CS, cokurt=CK, lamL=LL, MI=MI)

def offdiag_mean(M):
    N=M.shape[0]; return M[~np.eye(N,dtype=bool)].mean()
print("metric helpers defined")

metric helpers defined


In [7]:
X = rets.to_numpy()
emp = emp_pair_metrics(X)
rng = np.random.default_rng(1); G = rng.multivariate_normal(X.mean(0), np.cov(X.T), size=200000)
gau = emp_pair_metrics(G)
m = mps_pair_metrics(fit)
rows = []
rows.append(("total correlation (nats)", None, None, F.total_correlation(fit)))
rows.append(("mutual info (mean pair)", offdiag_mean(emp['MI']), offdiag_mean(gau['MI']), offdiag_mean(m['MI'])))
rows.append(("co-skewness (mean pair)", offdiag_mean(emp['coskew']), offdiag_mean(gau['coskew']), offdiag_mean(m['coskew'])))
rows.append(("co-kurtosis (mean pair)", offdiag_mean(emp['cokurt']), offdiag_mean(gau['cokurt']), offdiag_mean(m['cokurt'])))
rows.append(("lower-tail dep lamL (mean)", offdiag_mean(emp['lamL']), offdiag_mean(gau['lamL']), offdiag_mean(m['lamL'])))
rows.append(("tail asymmetry lamL-lamU", offdiag_mean(emp['lamL']-emp['lamU']), offdiag_mean(gau['lamL']-gau['lamU']), F.tail_dependence_asymmetry(fit)))
bat = pd.DataFrame(rows, columns=["metric","empirical","Gaussian","MPS"]).set_index("metric")
def winner(r):
    if r['empirical'] is None or np.isnan(r['empirical']): return ""
    return "MPS" if abs(r['MPS']-r['empirical']) < abs(r['Gaussian']-r['empirical']) else "Gaussian"
bat["closer_to_empirical"] = [winner(r) for _,r in bat.iterrows()]
print(bat.round(4).to_string())

                            empirical  Gaussian     MPS closer_to_empirical
metric                                                                     
total correlation (nats)          NaN       NaN  5.4690                    
mutual info (mean pair)        0.1061    0.0966  0.0936            Gaussian
co-skewness (mean pair)       -0.3318   -0.0023 -0.0001            Gaussian
co-kurtosis (mean pair)        6.7559    1.3243  1.7385                 MPS
lower-tail dep lamL (mean)     0.2953    0.2407  0.3700            Gaussian
tail asymmetry lamL-lamU       0.0320    0.0013  0.0542                 MPS


**Reading the battery.** The Gaussian column is what a covariance model implies: co-skewness ≈ 0,
co-kurtosis ≈ its Gaussian floor, tail dependence ≈ 0, tail asymmetry ≈ 0 — *by construction* it cannot
carry higher-order structure. Wherever the empirical value departs from Gaussian and the **MPS moves in the
same direction**, the MPS is capturing dependence covariance is blind to.

## 4 · Economic interpretation — WHICH pairs drive each effect (beyond statistics)
Averages hide the story. Below we surface the specific asset pairs that dominate each higher-order effect and
attach the economic mechanism.

In [8]:
def top_pairs(M, cols, k=5, sign="pos"):
    out=[]; N=len(cols)
    for i in range(N):
        for j in range(i+1,N): out.append((cols[i], cols[j], M[i,j]))
    if sign=="neg": out.sort(key=lambda t: t[2])                 # most negative first
    elif sign=="pos": out.sort(key=lambda t: t[2], reverse=True) # most positive first
    else: out.sort(key=lambda t: abs(t[2]), reverse=True)        # largest magnitude
    return out[:k]
cols=list(rets.columns)
print("TOP LOWER-TAIL-DEPENDENCE pairs (joint-crash clustering; economic: credit-equity, flight dynamics):")
for a,b,v in top_pairs(emp['lamL'], cols, sign="pos"): print("  %-6s %-6s  lamL=%.2f  [%s / %s]" % (a,b,v,CLASS[a],CLASS[b]))
print("\nMOST NEGATIVE CO-SKEW pairs (asset falls when the other's variance spikes; safe-haven / downside):")
sym = emp['coskew'] + emp['coskew'].T
for a,b,v in top_pairs(sym, cols, sign="neg"): print("  %-6s %-6s  coskew=%.3f  [%s / %s]" % (a,b,v,CLASS[a],CLASS[b]))
print("\nHIGHEST CO-KURTOSIS pairs (shared fat-tailed extremes / common liquidity shocks):")
for a,b,v in top_pairs(emp['cokurt'], cols, sign="pos"): print("  %-6s %-6s  cokurt=%.2f  [%s / %s]" % (a,b,v,CLASS[a],CLASS[b]))

TOP LOWER-TAIL-DEPENDENCE pairs (joint-crash clustering; economic: credit-equity, flight dynamics):
  EFA    USMKT   lamL=0.69  [Intl equity / US equity]
  TLT    LQD     lamL=0.67  [Rates / Credit]
  EFA    EEM     lamL=0.62  [Intl equity / Intl equity]
  HYG    USMKT   lamL=0.60  [Credit / US equity]
  NVDA   USMKT   lamL=0.57  [US equity / US equity]

MOST NEGATIVE CO-SKEW pairs (asset falls when the other's variance spikes; safe-haven / downside):
  VNQ    USMKT   coskew=-2.289  [Real asset / US equity]
  VNQ    EFA     coskew=-2.121  [Real asset / Intl equity]
  VNQ    EEM     coskew=-2.015  [Real asset / Intl equity]
  EFA    USMKT   coskew=-1.965  [Intl equity / US equity]
  EFA    EEM     coskew=-1.791  [Intl equity / Intl equity]

HIGHEST CO-KURTOSIS pairs (shared fat-tailed extremes / common liquidity shocks):
  VNQ    USMKT   cokurt=20.63  [Real asset / US equity]
  VNQ    HYG     cokurt=20.15  [Real asset / Credit]
  VNQ    EFA     cokurt=17.49  [Real asset / Intl equity]
 

## 5 · Nonlinear dependence — MI beyond what correlation implies
For a Gaussian pair, mutual information is exactly MI_gauss = −½ ln(1−ρ²). Any **excess** of empirical MI over
that is dependence the correlation coefficient cannot see (nonlinearity, tail coupling, regime mixing).

In [9]:
rho = np.corrcoef(X.to_numpy().T) if hasattr(X,"to_numpy") else np.corrcoef(X.T)  # reordered space (matches emp['MI'])
MIg = -0.5*np.log(np.clip(1-rho**2, 1e-9, 1))
excess = emp['MI'] - MIg
print("Mean excess MI (empirical - Gaussian-implied) = %.4f nats" % offdiag_mean(excess))
print("TOP NONLINEAR-DEPENDENCE pairs (excess MI over the linear-correlation implication):")
for a,b,v in top_pairs(excess, cols, sign="pos"):
    i,j=cols.index(a),cols.index(b)
    print("  %-6s %-6s  excessMI=%.3f  (rho=%.2f)  [%s / %s]" % (a,b,v,rho[i,j],CLASS[a],CLASS[b]))

Mean excess MI (empirical - Gaussian-implied) = -0.0048 nats
TOP NONLINEAR-DEPENDENCE pairs (excess MI over the linear-correlation implication):
  TLT    LQD     excessMI=0.231  (rho=0.69)  [Rates / Credit]
  CL     HYG     excessMI=0.042  (rho=0.21)  [Real asset / Credit]
  TLT    HYG     excessMI=0.040  (rho=0.00)  [Rates / Credit]
  TLT    VNQ     excessMI=0.034  (rho=0.01)  [Rates / Real asset]
  CL     USMKT   excessMI=0.030  (rho=0.24)  [Real asset / US equity]


## 6 · Regime-dependent correlation — does the MPS reproduce "diversification failure"?
We split days by **financial turbulence** (Mahalanobis distance, Kritzman-Li) into the calm 90% and the
stressed 10%, then measure average pairwise correlation in each. Empirically stress-corr ≫ calm-corr
(Longin-Solnik). A **Gaussian** model has *no* regime, so its stress/calm gap is ~0. If the **MPS** — sampled
from its fitted joint — shows the empirical stress/calm widening, it has captured regime structure covariance
cannot.

In [10]:
def turb_split_corr(Y, cov_inv, mu, thr):
    d = np.einsum("ij,jk,ik->i", Y-mu, cov_inv, Y-mu)
    hi = d >= thr; iu = np.triu_indices(Y.shape[1],1)
    return np.corrcoef(Y[~hi].T)[iu].mean(), np.corrcoef(Y[hi].T)[iu].mean()
mu = X.mean(0); cov_inv = np.linalg.pinv(np.cov(X.T))
dfull = np.einsum("ij,jk,ik->i", X-mu, cov_inv, X-mu); thr = np.quantile(dfull, 0.90)
e_calm, e_str = turb_split_corr(X, cov_inv, mu, thr)
g_calm, g_str = turb_split_corr(G[:X.shape[0]], cov_inv, mu, thr)
S = fit.sample_bins(X.shape[0]*3, seed=7)                      # sampled bin codes (n, N)
Smap = np.stack([fit.bin_values[k][S[:,k].astype(int)] for k in range(fit.n_sites)], 1)  # -> representative returns
m_calm, m_str = turb_split_corr(Smap, cov_inv, mu, thr)
reg = pd.DataFrame({"calm (90%)":[e_calm,g_calm,m_calm], "stress (10%)":[e_str,g_str,m_str]}, index=["empirical","Gaussian","MPS"])
reg["stress-calm gap"] = reg["stress (10%)"]-reg["calm (90%)"]
print(reg.round(3).to_string())
fig, ax = plt.subplots(figsize=(7,4))
reg[["calm (90%)","stress (10%)"]].plot(kind="bar", ax=ax); ax.set_ylabel("avg pairwise correlation")
ax.set_title("Regime-dependent correlation: does dependence rise in stress?"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(P("data/ca_regime.png"), dpi=110); plt.show()

           calm (90%)  stress (10%)  stress-calm gap
empirical       0.272         0.325            0.053
Gaussian        0.281         0.223           -0.058
MPS             0.199         0.234            0.035

## 7 · Deep-tail k-way co-crash — how many assets crash together
The headline tail-complexity diagnostic: probability that **k or more** assets are simultaneously in their
bottom-q tail on the same day — empirical vs Gaussian vs MPS. Gaussian tail-independence systematically
under-counts joint crashes; the MPS should track the empirical fat joint tail.

In [11]:
# rank-based (copula) tail: uniformize each asset so the MARGINAL exceedance is exactly q for all three
# distributions -> isolates DEPENDENCE (not marginal fit), and is robust to the MPS's discrete bins.
from scipy.stats import rankdata
def kway(sample, q=0.10):
    N = sample.shape[1]; n = len(sample)
    U = np.column_stack([rankdata(sample[:,k])/(n+1) for k in range(N)])
    below = (U <= q).sum(1)
    return np.array([(below >= k).mean() for k in range(1, N+1)])
e_k = kway(X); g_k = kway(G[:200000]); m_k = kway(Smap)
kk = np.arange(1, N+1)
fig, ax = plt.subplots(figsize=(9,5))
ax.semilogy(kk, e_k, "ko-", label="empirical (truth)"); ax.semilogy(kk, g_k, "b^--", label="Gaussian null"); ax.semilogy(kk, m_k, "gs-", label="MPS")
ax.set_xlabel("k = number of assets jointly in bottom-10% tail"); ax.set_ylabel("P(≥ k co-crash)  [log]")
ax.set_title("Deep-tail k-way co-crash — cross-asset"); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(P("data/ca_cocrash.png"), dpi=110); plt.show()
kdf = pd.DataFrame({"k":kk, "empirical":e_k, "Gaussian":g_k, "MPS":m_k,
                    "MPS/Gau@k":np.where(g_k>0, m_k/np.clip(g_k,1e-9,None), np.nan)}).set_index("k")
print(kdf.round(5).to_string())
khi = max(4, N//2)
print("\nAt k=%d: empirical=%.4f  Gaussian=%.4f  MPS=%.4f  (Gaussian under-counts by %.1fx)" %
      (khi, e_k[khi-1], g_k[khi-1], m_k[khi-1], e_k[khi-1]/max(g_k[khi-1],1e-9)))

    empirical  Gaussian      MPS  MPS/Gau@k
k                                          
1     0.47619   0.52124  0.48858    0.93733
2     0.25707   0.27601  0.26365    0.95521
3     0.16105   0.16162  0.15847    0.98052
4     0.10879   0.10170  0.10234    1.00630
5     0.07433   0.06520  0.06478    0.99360
6     0.05226   0.03963  0.03833    0.96713
7     0.03329   0.02148  0.01936    0.90118
8     0.01820   0.00936  0.00826    0.88239
9     0.01007   0.00296  0.00219    0.73991
10    0.00503   0.00070  0.00052    0.73219
11    0.00232   0.00008  0.00013    1.51822
12    0.00000   0.00000  0.00000    0.00000

At k=6: empirical=0.0523  Gaussian=0.0396  MPS=0.0383  (Gaussian under-counts by 1.3x)


## 8 · Verdict — statistical result first, economic reading only where it holds
Read this against the printed numbers above. The discipline: **state the statistical finding; attach an
economic interpretation only where the data actually supports one, and say so plainly when a result is purely
statistical.**

**Statistical findings (what the MPS shows that covariance cannot):**
- The **deep-tail k-way co-crash** curve is the core diagnostic — the Gaussian null is tail-*independent* by
  construction and under-counts joint crashes at high k; whether the MPS tracks the empirical fat tail is
  read directly off §7's table (report the k where the gap is largest).
- The **battery** (§3) shows which higher-order quantities the MPS moves toward empirical on (co-skew,
  co-kurt, λ_L, tail asymmetry, MI) vs where it does not — report the `closer_to_empirical` column honestly,
  including any metric where the Gaussian wins.
- **Nonlinear dependence** (§5): excess MI over the linear-correlation implication — a purely statistical
  quantity; flag the top pairs but only claim economics if they line up with a known mechanism.

**Where an economic reading is genuinely supported (check the top-pairs tables in §4–§6 before asserting):**
- *Credit–equity linkage* — if **HYG–(USMKT/AAPL/NVDA)** tops the λ_L table, high-yield is an equity proxy in
  the left tail despite its "bond" label (economic).
- *Safe-haven signature* — if **GC / TLT** carry the most negative co-skew with equities, that is the
  flight-to-quality effect (economic). If instead the negative co-skew is dominated by unrelated pairs, treat
  it as statistical only.
- *Crypto tail coupling* — if **BTC** appears in the high-co-kurtosis pairs, that is shared fat-tailed
  liquidity shocks (COVID-2020, 2022) (economic).
- *Regime dependence* (§6) — if empirical stress-corr ≫ calm-corr and the MPS reproduces the widening while
  the Gaussian gap is ~0, that is regime complexity (economic).

**Honest caveat.** Some diagnostics here (correlation length, compression ratio, reconstruction error, excess
MI on idiosyncratic pairs) are **structural/statistical** with no clean economic story — that is fine and is
noted as such. The features earn their place as *distributional diagnostics*; only a subset map to a market
mechanism, and we only claim the mechanism where the top-pairs evidence backs it.